In [13]:
import os
import pandas as pd
import torch
from torch import nn
from torchvision.io import decode_image, ImageReadMode
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

In [14]:
class IsingDataset(Dataset):
    def __init__(self, csv_path, data_root, split=None, transform=None):
        df = pd.read_csv(csv_path)
        if split is not None:
            df = df[df["split"] == split]
        self.df        = df.reset_index(drop=True)
        self.data_root = data_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.data_root, row["filepath"])
        image    = decode_image(img_path, mode=ImageReadMode.GRAY)  # → uint8 [1, H, W]
        label    = int(row["label"])
        T        = float(row["T"])
        L        = int(row["L"])
        if self.transform:
            image = self.transform(image)
        return image, label, T, L


In [15]:
# Transform pipeline:
#   ImageReadMode.GRAY already gives [1, H, W] uint8 — no Grayscale() needed
#   Resize    : → [1, 64, 64]
#   float32   : uint8 [0,255]  → float [0.0, 1.0]
#   Normalize : [0,1]          → [-1, 1]  (aligns with physical spin values ±1)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize((0.5,), (0.5,)),
])

DATA_ROOT = "data"
CSV_PATH  = "data/dataset.csv"

full_dataset = IsingDataset(CSV_PATH, DATA_ROOT, split="train",    transform=transform)
critical_set = IsingDataset(CSV_PATH, DATA_ROOT, split="critical", transform=transform)

n       = len(full_dataset)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

print(f"train: {len(train_set)}  |  val: {len(val_set)}  |  test: {len(test_set)}  |  critical: {len(critical_set)}")

train: 3301  |  val: 707  |  test: 708  |  critical: 2352


In [16]:
train_loader    = DataLoader(train_set,    batch_size=64, shuffle=True,  num_workers=0)
val_loader      = DataLoader(val_set,      batch_size=64, shuffle=False, num_workers=0)
test_loader     = DataLoader(test_set,     batch_size=64, shuffle=False, num_workers=0)
critical_loader = DataLoader(critical_set, batch_size=64, shuffle=False, num_workers=0)

In [17]:
class IsingNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu = nn.Sequential(
            nn.Linear(64 * 64, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # Binary classification
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu(x)
        return logits.squeeze(1)  # → [batch_size]

In [18]:
model = IsingNetwork()

#logits = model(train_set)

In [19]:
import torch.nn.functional as F

T_C = 2.269  # 2D Ising critical temperature (J/k_B units)

class IsingCNN(nn.Module):
    """
    Dual-head CNN for Ising configurations.
    Inputs : image [B,1,64,64], grid size L [B]
    Outputs: cls_logit [B]  (BCE loss → label 0/1)
             T_pred    [B]  (MSE loss → T / T_C, dimensionless)
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # 64x64 → 32x32
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            # 32x32 → 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            # 16x16 → 8x8
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            # 8x8 → 1x1
            nn.AdaptiveAvgPool2d(1),
        )
        # +1 for normalised grid size L
        self.cls_head = nn.Sequential(nn.Linear(129, 64), nn.ReLU(), nn.Linear(64, 1))
        self.reg_head = nn.Sequential(nn.Linear(129, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x, L):
        feat  = self.features(x).flatten(1)            # [B, 128]
        L_n   = (L.float() / 100.0).unsqueeze(1)       # normalise; adjust 100 to actual L_max
        feat  = torch.cat([feat, L_n], dim=1)          # [B, 129]
        return self.cls_head(feat).squeeze(1), self.reg_head(feat).squeeze(1)


In [20]:
device = torch.device("mps" if torch.backends.mps.is_available() else
                       "cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model     = IsingCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

CLS_W = 1.0   # weight for classification loss
REG_W = 1.0   # weight for regression loss (T/T_C scale ~1, so equal weighting is fine)


device: mps


In [21]:
def run_epoch(model, loader, optimizer=None):
    """
    One pass over `loader`.
    If optimizer is given → training mode; otherwise → eval mode.
    Returns dict of mean metrics over the epoch.
    """
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = total_cls = total_reg = correct = n = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for images, labels, T, L in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            T      = T.float().to(device)
            L      = L.to(device)

            cls_logit, T_pred = model(images, L)

            cls_loss = F.binary_cross_entropy_with_logits(cls_logit, labels)
            reg_loss = F.mse_loss(T_pred, T / T_C)
            loss     = CLS_W * cls_loss + REG_W * reg_loss

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs       = images.size(0)
            n       += bs
            total_loss += loss.item()     * bs
            total_cls  += cls_loss.item() * bs
            total_reg  += reg_loss.item() * bs
            correct    += ((cls_logit > 0) == labels.bool()).sum().item()

    return {
        "loss": total_loss / n,
        "cls_loss": total_cls / n,
        "reg_mse": total_reg / n,         # MSE in (T/T_C)^2 units
        "reg_mae_K": (total_reg / n) ** 0.5 * T_C,  # rough MAE estimate in J/k_B
        "acc": correct / n,
    }


In [22]:
EPOCHS = 20
best_val_loss = float("inf")

print(f"{'Ep':>3}  {'train_loss':>10}  {'train_acc':>9}  {'val_loss':>9}  {'val_acc':>8}  {'val_MAE':>8}")
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer)
    va = run_epoch(model, val_loader)

    scheduler.step(va["loss"])

    if va["loss"] < best_val_loss:
        best_val_loss = va["loss"]
        torch.save(model.state_dict(), "model_ian.pt")

    print(f"{epoch:3d}  {tr['loss']:10.4f}  {tr['acc']:9.4f}  {va['loss']:9.4f}  {va['acc']:8.4f}  {va['reg_mae_K']:8.4f}")


 Ep  train_loss  train_acc   val_loss   val_acc   val_MAE
  1      0.2228     0.9800     0.0768    0.9986    0.5480
  2      0.0231     0.9991     0.0222    1.0000    0.3089
  3      0.0172     1.0000     0.0133    1.0000    0.2493
  4      0.0190     0.9982     0.0333    1.0000    0.3965
  5      0.0158     0.9994     0.0121    1.0000    0.2419
  6      0.0118     1.0000     0.0168    1.0000    0.2880
  7      0.0131     0.9997     0.0171    1.0000    0.2832
  8      0.0119     1.0000     0.0096    1.0000    0.2191
  9      0.0126     0.9997     0.0110    1.0000    0.2338
 10      0.0117     1.0000     0.0100    1.0000    0.2249
 11      0.0111     0.9997     0.0166    0.9986    0.2741
 12      0.0098     1.0000     0.0165    1.0000    0.2895
 13      0.0088     1.0000     0.0082    1.0000    0.2027
 14      0.0086     0.9997     0.0101    1.0000    0.2234
 15      0.0080     1.0000     0.0088    1.0000    0.2116
 16      0.0080     0.9997     0.0258    1.0000    0.3626
 17      0.007

In [23]:
# Load best checkpoint and evaluate on held-out sets
model.load_state_dict(torch.load("model_ian.pt", map_location=device))

test_m     = run_epoch(model, test_loader)
critical_m = run_epoch(model, critical_loader)

print("=== Test set ===")
print(f"  accuracy : {test_m['acc']:.4f}")
print(f"  T MAE    : {test_m['reg_mae_K']:.4f} J/k_B")

print("\n=== Critical set (T ≈ T_C) ===")
print(f"  accuracy : {critical_m['acc']:.4f}  ← expect lower; configs are ambiguous near T_C")
print(f"  T MAE    : {critical_m['reg_mae_K']:.4f} J/k_B")


=== Test set ===
  accuracy : 1.0000
  T MAE    : 0.2092 J/k_B

=== Critical set (T ≈ T_C) ===
  accuracy : 0.8912  ← expect lower; configs are ambiguous near T_C
  T MAE    : 0.2868 J/k_B
